# 📊 01 — Baseline Evaluation

**Tujuan:** Mengukur keyword refusal rate (RR) pada model asli sebelum intervensi.

**Lokasi:** VS Code (ringan, muat di RTX 2050 4GB)

---
**Output:** `results/baseline_generations.csv` + baris pertama `results/metrics.csv`

Saved results are in `results/reference/`. Your results may differ with other inputs or settings. See `docs/results.md` for details.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src" / "config.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from inside the project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
# ─── Setup ───
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import pandas as pd
from transformer_lens import HookedTransformer

from src.config import DEFAULT_MODEL, DEVICE, N_INST_TRAIN, N_INST_TEST, SEED, LOGS_DIR, RESULTS_DIR
from src.data_loader import get_harmful_instructions, get_harmless_instructions, create_holdout_set, get_train_test_split
from src.generation_utils import tokenize_instructions_fn, get_generations
from src.metrics import compute_refusal_rate, compute_asr, classify_refusal

torch.manual_seed(SEED)
print(f"Device: {DEVICE}")
print(f"Model: {DEFAULT_MODEL}")

print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GB")
print(f"Device: {DEVICE}")

In [ ]:
print(f"Python: {sys.executable}")
print(f"Torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# ─── Load model ───
model = HookedTransformer.from_pretrained_no_processing(
    DEFAULT_MODEL,
    dtype=torch.bfloat16,
    default_padding_side='left',
    device=DEVICE,
)
model.tokenizer.padding_side = 'left'
model.tokenizer.pad_token = model.tokenizer.eos_token
print(f"Model loaded: {model.cfg.model_name}")
print(f"n_layers: {model.cfg.n_layers}, d_model: {model.cfg.d_model}")

print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GB")
print(f"Device: {DEVICE}")

In [ ]:
# ─── Load data ───
harmful = get_harmful_instructions()
harmless = get_harmless_instructions()
print(f"Harmful: {len(harmful)}, Harmless: {len(harmless)}")

# Buat holdout (tidak dipakai untuk training refusal direction)
holdout_harmful, holdout_harmless = create_holdout_set(harmful, harmless)
print(f"Holdout set: {len(holdout_harmful)} harmful, {len(holdout_harmless)} harmless")

# Train/test split
harmful_train, harmless_train, harmful_test, harmless_test = get_train_test_split(
    harmful, harmless, holdout_harmful, holdout_harmless,
    n_train=N_INST_TRAIN, n_test=N_INST_TEST
)
print(f"Train: {len(harmful_train)} harmful + {len(harmless_train)} harmless")
print(f"Test:  {len(harmful_test)} harmful + {len(harmless_test)} harmless")

In [ ]:
# ─── Generate baseline (harmful) ───
print("Generating baseline (harmful prompts)...")
baseline_harmful = get_generations(
    model, harmful_test, tokenize_instructions_fn,
    fwd_hooks=[], max_tokens_generated=150
)

# Tampilkan beberapa contoh
for i in range(min(3, len(baseline_harmful))):
    print(f"\n--- Prompt {i} ---")
    print(f"Q: {harmful_test[i]}")
    print(f"A: {baseline_harmful[i][:300]}")
    print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    print(f"Reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GB")
    print(f"Device: {DEVICE}")

In [ ]:
# ─── Hitung metrik baseline ───
baseline_rr = compute_refusal_rate(baseline_harmful)
baseline_asr = compute_asr(baseline_harmful, harmful_test)

print(f"BASELINE METRICS")
print(f"Refusal Rate (RR):  {baseline_rr:.2%}")
print(f"ASR heuristic: {baseline_asr:.2%}")
print(f"Sample size: {len(baseline_harmful)}")

In [ ]:
# ─── Generate baseline (harmless) untuk inspeksi kualitatif ───
print("Generating baseline (harmless prompts)...")
baseline_harmless = get_generations(
    model, harmless_test, tokenize_instructions_fn,
    fwd_hooks=[], max_tokens_generated=150
)

for i in range(min(2, len(baseline_harmless))):
    print(f"\n--- Harmless {i} ---")
    print(f"Q: {harmless_test[i]}")
    print(f"A: {baseline_harmless[i][:200]}")

In [ ]:
print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GB")
print(f"Device: {DEVICE}")

In [ ]:
# ─── Simpan hasil ───
# Log generasi
df_harmful = pd.DataFrame({
    'prompt': harmful_test,
    'generation': baseline_harmful,
    'refused': [classify_refusal(g) for g in baseline_harmful],
    'condition': 'baseline'
})
df_harmful.to_csv(LOGS_DIR / 'baseline_harmful.csv', index=False)

df_harmless = pd.DataFrame({
    'prompt': harmless_test,
    'generation': baseline_harmless,
    'condition': 'baseline'
})
df_harmless.to_csv(LOGS_DIR / 'baseline_harmless.csv', index=False)

# Metrics ke CSV (append)
metrics_path = RESULTS_DIR / 'metrics.csv'
df_metrics = pd.DataFrame([{
    'condition': 'baseline',
    'refusal_rate': baseline_rr,
    'asr_heuristic': baseline_asr,
    'n_samples': len(baseline_harmful),
    'model': DEFAULT_MODEL,
}])
df_metrics.to_csv(metrics_path, index=False, mode='w')  # overwrite, nanti di-append
print(f"\nSaved: {metrics_path}")
print(df_metrics)

---
**Next:** Lanjut ke `02_extract_activations.ipynb` untuk mengekstrak refusal direction.